In [67]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 3.1/12.8 MB 30.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 42.3 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [68]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [69]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [70]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [71]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [72]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [73]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [74]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [ ]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti (osebe, kraji, jeziki, narodi)
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    # ustvarimo množico besed, ki so del prepoznanih entitet (imen)
    imena_v_tekstu = {ent.text.lower() for ent in doc.ents if ent.label_ in odstrani_entitete}

    for token in doc:
        # če je beseda ime (NER)
        if token.text.lower() in imena_v_tekstu:
            continue
            
        # spaCy uporablja oznake: NOUN ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            
            # samo črke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [164]:
import random
from sklearn.feature_extraction import text

In [165]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    'book', 'novel', 'story', 'reader', 'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series', 'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face', 'place', 'mean', 'text', 'author', 'original', 
    'u', 'seller', 'masterpiece', 'literature', 'best', 'read', 'man', 'men', 
    'woman', 'life', 'fiction', 'tale', 'character', 'page', 'write', 'writer',
    'chapter', 'volume', 'collection', 'everything', 'day', 'world', 'come', 'series',
    'know', 'want', 'come', 'dragomir', 'tessa', 'dimitri', 'strigoi','lissa', 'sit', 'say', 'isbn',
    'bestselle', 'review'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [167]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\najbolj_popularne\naj_ang_opisi\*.txt')[:250]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.6)

In [168]:
X = vectorizer.fit_transform(filepaths) 


In [169]:
words = vectorizer.get_feature_names_out()
freq = np.asarray(X.sum(axis=0)).flatten()

top_words = [words[i] for i in freq.argsort()[-50:]]
print(top_words)

['end', 'crime', 'killer', 'hogwart', 'power', 'city', 'perfect', 'father', 'true', 'modern', 'school', 'daughter', 'thing', 'powerful', 'earth', 'century', 'generation', 'beautiful', 'truth', 'age', 'alternate', 'evil', 'night', 'human', 'mother', 'murder', 'home', 'town', 'people', 'heart', 'house', 'war', 'death', 'adventure', 'child', 'sister', 'way', 'dark', 'past', 'secret', 'friend', 'great', 'good', 'boy', 'girl', 'young', 'little', 'old', 'family', 'love']


In [170]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5580 stored elements and shape (250, 880)>
  Coords	Values
  (0, 288)	1
  (0, 101)	1
  (0, 177)	3
  (0, 382)	2
  (0, 301)	3
  (0, 349)	1
  (0, 161)	1
  (0, 450)	1
  (0, 79)	1
  (0, 315)	1
  (0, 15)	1
  (0, 273)	1
  (0, 538)	1
  (0, 710)	1
  (0, 123)	1
  (0, 174)	1
  (0, 758)	2
  (0, 687)	1
  (0, 520)	1
  (0, 111)	1
  (0, 378)	1
  (0, 464)	1
  (1, 464)	2
  (1, 879)	1
  (1, 255)	1
  :	:
  (249, 11)	1
  (249, 331)	1
  (249, 212)	1
  (249, 444)	2
  (249, 36)	1
  (249, 715)	1
  (249, 344)	1
  (249, 30)	1
  (249, 532)	1
  (249, 365)	2
  (249, 868)	1
  (249, 870)	1
  (249, 341)	1
  (249, 808)	1
  (249, 250)	1
  (249, 312)	1
  (249, 437)	1
  (249, 869)	1
  (249, 41)	1
  (249, 393)	1
  (249, 617)	2
  (249, 451)	1
  (249, 68)	1
  (249, 60)	1
  (249, 525)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [1 0 2 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   able  absolute  academy  accident  acclaimed  account  achie

In [171]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [177]:
# test 

W, H, errors = nmf(X, 6)
print(errors)
#print(W)
#print(H)

[np.float64(89.15109415908486), np.float64(88.25892509151423), np.float64(87.37643393236836), np.float64(86.53845094849775), np.float64(85.90883519294343), np.float64(85.47186148884741), np.float64(85.15704681722708), np.float64(84.92818497247622), np.float64(84.75603039315203), np.float64(84.62195839105802), np.float64(84.51384625771581), np.float64(84.4179354055396), np.float64(84.32176789280545), np.float64(84.21507980420756), np.float64(84.09485400933673), np.float64(83.96942741144214), np.float64(83.85472920744122), np.float64(83.76535250740214), np.float64(83.70055178891643), np.float64(83.65289892798899), np.float64(83.61451983874112), np.float64(83.58339365820696), np.float64(83.55656609782757), np.float64(83.53449795518313), np.float64(83.51604091710172), np.float64(83.5004201864292), np.float64(83.4872372992084), np.float64(83.47604924074791), np.float64(83.4663098393779), np.float64(83.45763672712205), np.float64(83.4499209817887), np.float64(83.44289099382824), np.float64(8

In [178]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx+1}: {' '.join(top_words)}")

Tema 1: question mysterious hogwart truth power horror apartment night way young great adventure past secret dark
Tema 2: justice crime human land room people night killer child death mother family sister old girl
Tema 3: ready college beginning happy home young baby money farm big wood town little family house
Tema 4: bad debut heart perfect love school human evil princess true grace thriller vampire friend good
Tema 5: thing infernal son century devotion great extraordinary plan king alternate art heart young war love
Tema 6: fate secret island terror self sea half bear room stranger guest rhyme murder boy little


In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

def prikazi_wordclouds(H, feature_names, n_top_words=10):
    n_topics = H.shape[0]
    
    
    plt.figure(figsize=(20, 10))
    
    for topic_idx, topic in enumerate(H):
        # slovar {beseda: utež} za WordCloud
        # top n besed za vsako temo
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        topic_words = {feature_names[i]: topic[i] for i in top_features_ind}
        
        
        wc = WordCloud(width=400, height=400, background_color='white', 
                       colormap='viridis', max_words=n_top_words)
        wc.generate_from_frequencies(topic_words)
    
        plt.subplot(2, 2, topic_idx + 1)
        plt.imshow(wc, interpolation='bilinear')
        plt.title(f"Tema {topic_idx + 1}", fontsize=20)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

feature_names = vectorizer.get_feature_names_out()
prikazi_wordclouds(H, feature_names)

In [183]:
import pandas as pd

# Matrika W nam pove, kateri temi pripada katera knjiga
# Dobimo indeks najvišje vrednosti v vsaki vrstici
dominant_topics = np.argmax(W, axis=1) + 1 # +1 da začnemo s Tema 1

# Pripravimo seznam imen datotek (brez celotne poti, samo ime)
import os
filenames = [os.path.basename(f) for f in filepaths]

# Ustvarimo tabelo
df_results = pd.DataFrame({
    'Knjiga': filenames,
    'Dominantna Tema': dominant_topics
})

# Izpišemo prvih 10 rezultatov
print(df_results.head(50))

# Če želiš preveriti specifično temo (npr. vse knjige v Temi 2):
print(df_results[df_results['Dominantna Tema'] == 3])

          Knjiga  Dominantna Tema
0     opis_1.txt                2
1    opis_10.txt                5
2   opis_100.txt                1
3   opis_101.txt                2
4   opis_102.txt                2
5   opis_103.txt                4
6   opis_104.txt                2
7   opis_105.txt                4
8   opis_106.txt                4
9   opis_107.txt                1
10  opis_108.txt                1
11  opis_109.txt                2
12   opis_11.txt                5
13  opis_110.txt                4
14  opis_111.txt                5
15  opis_112.txt                4
16  opis_113.txt                2
17  opis_114.txt                6
18  opis_115.txt                1
19  opis_116.txt                5
20  opis_117.txt                2
21  opis_118.txt                2
22  opis_119.txt                2
23   opis_12.txt                5
24  opis_120.txt                1
25  opis_121.txt                4
26  opis_122.txt                1
27  opis_123.txt                1
28  opis_124.t